In [1]:
# NLP Text Clustering with K-Means — Tort1024.csv

## 1. Import libraries

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
```

## 2. Download NLTK resources

Run this cell once:

```python
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
```

## 3. Load the dataset

```python
df = pd.read_csv("Tort1024.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())
```

## 4. Check the dataset

```python
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())
```

## 5. Select the text column

First, identify which column contains the text:

```python
for column in df.columns:
    print(f"{column}:")
    print(df[column].astype(str).head(2).tolist())
    print()
```

After checking the output, set the text column below.

```python
TEXT_COLUMN = "text"   # Change "text" to the actual text column name

df = df.dropna(subset=[TEXT_COLUMN]).copy()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

print("Number of documents:", len(df))
```

## 6. NLP preprocessing

The following function:

* converts text to lowercase
* removes URLs
* removes punctuation and numbers
* removes stopwords
* lemmatises words

```python
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # Keep alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    # Tokenisation
    words = text.split()
    
    # Stopword removal and lemmatisation
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]
    
    return " ".join(words)

df["clean_text"] = df[TEXT_COLUMN].apply(preprocess_text)

display(df[[TEXT_COLUMN, "clean_text"]].head())
```

## 7. Convert text into TF-IDF features

TF-IDF converts the text into numerical features that can be used by K-Means.

```python
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X.shape)
```

## 8. Find a suitable number of clusters

We can test several values of `K` using the silhouette score.

```python
scores = []

K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X)
    
    score = silhouette_score(X, labels)
    scores.append(score)
    
    print(f"K = {k}, Silhouette Score = {score:.4f}")
```

## 9. Plot the silhouette scores

```python
plt.figure(figsize=(8, 5))

plt.plot(
    list(K_range),
    scores,
    marker="o"
)

plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Selecting K using Silhouette Score")
plt.xticks(list(K_range))
plt.grid(True)

plt.show()
```

## 10. Select K

You can automatically select the K with the highest silhouette score:

```python
best_k = list(K_range)[np.argmax(scores)]

print("Selected number of clusters:", best_k)
```

If you already know how many categories/classes you expect, you can manually specify it instead:

```python
# Example:
# best_k = 5
```

## 11. Train K-Means

```python
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print(df["cluster"].value_counts().sort_index())
```

## 12. Examine the most important words in each cluster

```python
terms = vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster_number in range(best_k):
    top_words = [
        terms[ind]
        for ind in order_centroids[cluster_number, :15]
    ]
    
    print(f"\nCluster {cluster_number}")
    print("Top words:", ", ".join(top_words))
```

## 13. Display example documents from each cluster

```python
for cluster_number in range(best_k):
    
    print("\n" + "=" * 80)
    print(f"CLUSTER {cluster_number}")
    print("=" * 80)
    
    cluster_documents = df[
        df["cluster"] == cluster_number
    ][TEXT_COLUMN].head(5)
    
    for i, document in enumerate(cluster_documents, 1):
        print(f"\nDocument {i}:")
        print(document[:1000])
```

## 14. Visualise the clusters

K-Means operates in a high-dimensional TF-IDF space, so we can use PCA to visualise the clusters in two dimensions.

```python
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=2,
    random_state=42
)

X_2d = svd.fit_transform(X)

plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["cluster"],
    alpha=0.6
)

plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("K-Means Clusters of Tort1024 Dataset")

plt.colorbar(scatter, label="Cluster")

plt.show()
```

## 15. Add cluster labels to the dataset

```python
df[["cluster", TEXT_COLUMN]].head(20)
```

## 16. Save the clustered dataset

```python
output_file = "Tort1024_kmeans_results.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Saved results to: {output_file}")
```

## 17. Summary of the clustering

```python
print("Number of documents:", len(df))
print("Number of clusters:", best_k)
print("Number of TF-IDF features:", X.shape[1])

print("\nDocuments per cluster:")
print(df["cluster"].value_counts().sort_index())

print("\nFinal silhouette score:")
print(round(silhouette_score(X, df["cluster"]), 4))
```

# Complete workflow

The workflow used is:

**Tort1024.csv → Text preprocessing → TF-IDF → K-Means → Cluster analysis → Visualisation**

The important stages are:

1. Load `Tort1024.csv`.
2. Identify the column containing the text.
3. Clean and preprocess the text using NLP.
4. Convert the documents into TF-IDF vectors.
5. Test different values of `K`.
6. Select `K` using the silhouette score.
7. Apply K-Means clustering.
8. Identify the most important words in each cluster.
9. Examine example documents from each cluster.
10. Visualise the clusters.
11. Save the results as `Tort1024_kmeans_results.csv`.


SyntaxError: invalid character '→' (U+2192) (3463556611.py, line 312)